# Stage-1 Proxy Classifier — Exploratory Evaluation

Run this **after** `src/extract_embeddings.py` has produced `data/embeddings/coswara.npz` and
`data/embeddings/virufy.npz`, and after `src/train_classifier.py` has produced a trained model.

Purpose: sanity-check the embeddings actually separate the two classes at all (via t-SNE/PCA
visualization) before trusting any classifier metric — a classifier can report a misleadingly
good number on a small/imbalanced split even when the underlying embedding space doesn't
separate the classes in any meaningful way. This notebook is the visual gut-check for that.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import joblib

coswara = np.load('../data/embeddings/coswara.npz', allow_pickle=True)
X, y = coswara['embeddings'], coswara['labels']
print(f"Loaded {len(y)} samples: {y.sum()} positive, {(y==0).sum()} negative")

In [ ]:
# PCA first — cheap, gives a rough sense of separability before the more expensive t-SNE
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7, 6))
plt.scatter(X_pca[y==0, 0], X_pca[y==0, 1], alpha=0.4, label='Negative (healthy)', s=10)
plt.scatter(X_pca[y==1, 0], X_pca[y==1, 1], alpha=0.4, label='Positive (TB/COVID)', s=10)
plt.legend()
plt.title('PCA of YAMNet embeddings (Coswara)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
plt.show()

print(
    "\nIf the two classes look like a fully overlapping blob here, that is a meaningful "
    "signal (not necessarily a failure) — PCA is a linear, 2D projection of a 1024-dim space, "
    "so real separability can still exist in dimensions PCA doesn't show. Treat this as a "
    "first-pass sanity check, not a final verdict; the t-SNE plot below and the actual "
    "classifier's held-out AUC are more informative."
)

In [ ]:
# t-SNE — slower, but often reveals non-linear cluster structure PCA misses.
# Subsample if the dataset is large; t-SNE scales poorly past a few thousand points.
n_sample = min(2000, len(y))
idx = np.random.choice(len(y), n_sample, replace=False)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X[idx])
y_sample = y[idx]

plt.figure(figsize=(7, 6))
plt.scatter(X_tsne[y_sample==0, 0], X_tsne[y_sample==0, 1], alpha=0.4, label='Negative', s=10)
plt.scatter(X_tsne[y_sample==1, 0], X_tsne[y_sample==1, 1], alpha=0.4, label='Positive', s=10)
plt.legend()
plt.title('t-SNE of YAMNet embeddings (Coswara)')
plt.show()

In [ ]:
# Load the trained classifier and check its held-out performance once more, alongside the
# visual check above, before drawing any conclusion about whether Stage 1 "works."
pipeline_obj = joblib.load('../models/stage1_proxy_v1.pkl')
print('Model version:', pipeline_obj['model_version'])
print('Internal test AUC:', pipeline_obj['internal_test_auc'])
print('Holdout results:', {k: v['auc'] for k, v in pipeline_obj['holdout_results'].items()})
print('\nDisclaimer embedded in the model artifact itself:')
print(pipeline_obj['IMPORTANT_DISCLAIMER'])

## What to actually conclude from this notebook

Good separability + good held-out AUC here means: **the pipeline works end-to-end, and the
Stage-1 proxy task (TB/COVID vs. healthy cough) has a detectable acoustic signal** — exactly the
technical proof-of-pipeline goal from PRD §6 Phase 0. It says **nothing** about whether
silicosis has any acoustic signature at all. That question is untouched by anything in this
notebook and can only be answered with Phase 3's real paired clinical data.